# Multimodal Physiological Signal Analysis — Demo Walkthrough

An end-to-end tour of this repository's pipeline:

1. **Signal conditioning** — filtering, resampling, windowing (WESAD)
2. **Feature extraction** — time/frequency-domain descriptors
3. **Model** — per-modality CNN encoders → cross-modal GNN → classifier, with self-supervised pretraining

> **Provenance note.** This repo is a clean-room *reference implementation* reconstructing the
> approach of a 2025 University at Albany research internship. The original research code is
> unavailable; results quoted in the README (92% accuracy, +31% cross-modal feature
> extraction) are the reported outcomes of that research. Cells below that need the WESAD
> dataset are marked — everything else runs standalone on synthetic data.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
print("numpy", np.__version__)

## 1. The WESAD dataset

WESAD (Schmidt et al., ICMI 2018) contains chest-worn (RespiBAN @ 700 Hz: ECG, EDA, EMG,
respiration, temperature, accel) and wrist-worn (Empatica E4) recordings from 15 subjects,
labelled with affective states: `1 = baseline`, `2 = stress`, `3 = amusement`
(`0 = transient/undefined`, discarded). Access requires a signed EULA — see the README.

This cell checks whether the dataset is present:

In [ ]:
from data_loader import WESADLoader, LABEL_NAMES

DATA_DIR = os.path.abspath("../data/WESAD")
if os.path.isdir(DATA_DIR):
    loader = WESADLoader(DATA_DIR)
    print("subjects on disk:", loader.subjects)
    print("label map:", LABEL_NAMES)
else:
    print(f"dataset not found at {DATA_DIR} — remaining cells use synthetic data.")

## 2. Signal conditioning (synthetic demo — runs without the dataset)

We mimic a noisy ECG trace (heartbeat + 50 Hz mains hum + sensor noise), then apply the
same band-pass recipe `preprocessing.filter_modality` uses for real ECG data:

In [ ]:
from preprocessing import butter_bandpass, apply_filter

fs = 700  # RespiBAN chest sampling rate
t = np.arange(0, 5, 1 / fs)
rng = np.random.default_rng(0)
heartbeat = np.sin(2 * np.pi * 1.2 * t) + 0.3 * np.sin(2 * np.pi * 2.4 * t)
noisy = heartbeat + 0.5 * np.sin(2 * np.pi * 50 * t) + 0.4 * rng.standard_normal(len(t))

b, a = butter_bandpass(0.5, 40.0, fs)
clean = apply_filter(noisy, b, a)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[:700], noisy[:700], alpha=0.45, label="raw (noisy)")
ax.plot(t[:700], clean[:700], label="band-pass filtered 0.5–40 Hz")
ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude"); ax.legend();
plt.show()

## 3. Hand-crafted features

`features.py` computes time-domain descriptors (mean, RMS, Hjorth parameters, …),
frequency-domain descriptors (Welch band powers, spectral entropy), and simple HRV
metrics for cardiac signals — useful as an interpretable baseline alongside the
learned CNN embeddings:

In [ ]:
from features import time_domain_features, frequency_domain_features

td = time_domain_features(clean)
fd = frequency_domain_features(clean, fs)
for k, v in {**td, **fd}.items():
    print(f"{k:20s} {v:10.4f}")

## 4. Model architecture

```
 window (B, M, T)          M = modalities (ECG, EDA, EMG, Resp), T = time steps
       │
       ▼
 ┌─────────────┐   one 1D-CNN per modality → 128-d embedding each
 │ CNN encoders │
 └─────────────┘
       │  (B, M, 128)
       ▼
 ┌─────────────┐   nodes = modalities, edges = LEARNED affinities
 │ Cross-modal  │   2-layer GCN → mean pool over nodes
 │ GNN fusion   │
 └─────────────┘
       │  (B, 128) fused
       ├──► classifier → affective state (baseline / stress / amusement)
       └──► SSL projection head (contrastive pretraining only)
```

**Self-supervised pretraining:** two augmented views (jitter / scaling / temporal shift)
of each window are pushed together in projection space with the NT-Xent loss *before*
any labels are used — this is what warms up the cross-modal representations.

In [ ]:
try:
    import torch
    from models import MultimodalAffectNet

    model = MultimodalAffectNet(modalities=["ECG", "EDA", "EMG", "Resp"])
    x = torch.randn(4, 4, 3840)  # (batch, modalities, time) — 60 s @ 64 Hz
    print("logits shape:", tuple(model(x).shape))          # expect (4, 3)
    print("fused shape :", tuple(model.encode(x).shape))   # expect (4, 128)
    print("n params    :", sum(p.numel() for p in model.parameters()))
except ImportError:
    print("torch / torch_geometric not installed — run: pip install -r ../requirements.txt")

## 5. Training (from a terminal, with WESAD downloaded)

```bash
# contrastive pretraining of encoders + GNN (no labels needed)
python -m src.train --data-dir data/WESAD --phase ssl --epochs 50

# supervised fine-tuning, warm-started from the SSL checkpoint
python -m src.train --data-dir data/WESAD --phase supervised \
    --ssl-checkpoint checkpoints/ssl_best.pt --epochs 100

# evaluate on held-out subjects (default: S16, S17)
python -m src.evaluate --data-dir data/WESAD \
    --checkpoint checkpoints/supervised_best.pt --plot outputs/confusion_matrix.png
```

Splits are strictly **subject-wise** — no subject appears in both train and test,
which is the honest way to evaluate physiological models.

## 6. Results (reported outcomes of the original research)

| Result | Detail |
|---|---|
| **92% accuracy** | classification accuracy on held-out multimodal data |
| **+31% cross-modal feature extraction** | gain from GNN fusion + self-supervised pretraining over the encoder baseline |

These numbers describe the original 2025 study, not this reconstruction — see the
provenance note at the top. Re-running the protocol above on WESAD is the way to
validate them independently.

## References

- P. Schmidt, A. Reiss, R. Duerichen, C. Marberger, K. Van Laerhoven, "Introducing WESAD,
  a Multimodal Dataset for Wearable Stress and Affect Detection", *Proc. ICMI*, 2018.
- Dataset access (signed EULA required): https://ubi29.informatik.uni-siegen.de/usi/data_wesad.html